## Google Colab Full-Run Setup

This GPU notebook is configured for a full run.

Run cells in order:
1. Install dependencies + verify GPU.
2. Upload `eng.train`, `eng.testa`, `eng.testb`.
3. Run all remaining cells to generate metrics, shot summary, logs, and timing artifacts.

In [ ]:
import os
import subprocess
import sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Install dependencies (works on Colab and locally)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers",
        "torch",
        "sentencepiece",
        "pandas",
        "numpy",
        "tqdm",
    ],
    check=False,
)

import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("No GPU detected. In Colab: Runtime -> Change runtime type -> T4/A100 GPU.")

GPU available: False
Running on CPU — consider switching to a GPU runtime on Colab.


## Upload CoNLL Files (Colab)

Use this once to upload exactly these 3 files:
- `eng.train`
- `eng.testa`
- `eng.testb`

They will be moved into `/content/conll2003` automatically.

In [ ]:
from pathlib import Path
import shutil

required_files = ["eng.train", "eng.testa", "eng.testb"]
colab_conll_dir = Path("/content/conll2003")
colab_conll_dir.mkdir(parents=True, exist_ok=True)

in_colab = False
try:
    import google.colab  # type: ignore
    from google.colab import files  # type: ignore

    in_colab = True
except Exception:
    in_colab = False

if in_colab:
    print("Please upload: eng.train, eng.testa, eng.testb")
    _ = files.upload()

    for name in required_files:
        src = Path(name)
        dst = colab_conll_dir / name
        if src.exists():
            if dst.exists():
                dst.unlink()
            shutil.move(str(src), str(dst))

    missing = [n for n in required_files if not (colab_conll_dir / n).exists()]
    if missing:
        raise FileNotFoundError(f"Missing files after upload: {missing}")

    print("Dataset ready at:", colab_conll_dir)
    print("Files:", [p.name for p in sorted(colab_conll_dir.glob("*"))])
else:
    print("Not running in Colab. Skipping upload step.")

# Assignment 4: Prompt Engineering for NER

**Student:** MingHsiang Lee  
**Task:** Named Entity Recognition (NER) with local CoNLL-2003

## 0-shot vs Few-shot

- **0-shot**: instruction + input only, no examples.
- **Few-shot**: instruction + a few labeled examples + input.

This notebook compares both approaches.

In [ ]:
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Full-run defaults for Colab GPU
USE_FAST_MODE = False
FAST_SENTENCE_LIMIT = 80
FULL_SENTENCE_LIMIT = None  # Keep None to evaluate the full eng.testa split.

# Two-model comparison for assignment deliverable.
# T4-safe default: base + large. If OOM, switch to small + base.
MODEL_NAMES = [
    "google/flan-t5-base",
    "google/flan-t5-large",
    # "google/flan-t5-xl",  # Optional on larger GPUs
]

# Beam search width (higher = better structure, slower runtime).
NUM_BEAMS = 4
MAX_NEW_TOKENS = 256

# Save all artifacts here.
OUTPUT_DIR = Path("/content/assignment4_outputs") if Path("/content").exists() else Path("assignment4_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Colab-first dataset paths, then local fallbacks.
CONLL_REL_PATHS = [
    Path("/content/conll2003"),
    Path("/content/Assignment2_Name_Entity_Recognition/conll2003"),
    Path("Assignment2_Name_Entity_Recognition/conll2003"),
    Path("../Assignment2_Name_Entity_Recognition/conll2003"),
]

CONLL_DIR = None
for p in CONLL_REL_PATHS:
    if p.exists():
        CONLL_DIR = p
        break

if CONLL_DIR is None:
    raise FileNotFoundError(
        "Could not find CoNLL data. Upload eng.train/eng.testa/eng.testb or place dataset in a known path."
    )

EVAL_FILE = "eng.testa"
print("Using CoNLL directory:", CONLL_DIR)
print("Output directory:", OUTPUT_DIR)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")
print("Mode:", "FAST" if USE_FAST_MODE else "FULL")
print("Models:", MODEL_NAMES)

/Users/Thomas/Desktop/Nat_Lang_Engr_Meth_Tools/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using CoNLL directory: ../Assignment2_Name_Entity_Recognition/conll2003
Device: CPU


In [ ]:
def read_conll(file_path: Path):
    rows = []
    tokens, tags = [], []
    with file_path.open("r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                if tokens:
                    rows.append({"tokens": tokens, "tags": tags})
                    tokens, tags = [], []
                continue
            if line.startswith("-DOCSTART-"):
                continue
            parts = line.split()
            if len(parts) >= 4:
                tokens.append(parts[0])
                tags.append(parts[-1])
    if tokens:
        rows.append({"tokens": tokens, "tags": tags})
    return rows


def bio_to_entities(tokens, tags):
    out = []
    i = 0
    while i < len(tags):
        tag = tags[i]
        if tag == "O" or "-" not in tag:
            i += 1
            continue
        prefix, label = tag.split("-", 1)
        if prefix not in {"B", "I"}:
            i += 1
            continue
        ent = [tokens[i]]
        i += 1
        while i < len(tags) and tags[i] == f"I-{label}":
            ent.append(tokens[i])
            i += 1
        out.append({"entity": " ".join(ent), "type": label})
    return out


sentences = read_conll(CONLL_DIR / EVAL_FILE)
full_sentence_count = len(sentences)

data = []
for s in sentences:
    text = " ".join(s["tokens"])
    gold_entities = bio_to_entities(s["tokens"], s["tags"])
    data.append({"text": text, "gold_entities": gold_entities})

eval_df = pd.DataFrame(data)

if USE_FAST_MODE:
    eval_df = eval_df.head(FAST_SENTENCE_LIMIT).copy()
elif FULL_SENTENCE_LIMIT is not None:
    eval_df = eval_df.head(FULL_SENTENCE_LIMIT).copy()

print("Mode:", "FAST" if USE_FAST_MODE else "FULL")
print("Full sentence count:", full_sentence_count)
print("Evaluated sentence count:", len(eval_df))
display(eval_df.head(3))

Mode: FAST
Sentence count: 80


,text,gold_entities
0,CRICKET - LEICESTERSHIRE TAKE OVER AT TOP AFTE...,"[{'entity': 'LEICESTERSHIRE', 'type': 'ORG'}]"
1,LONDON 1996-08-30,"[{'entity': 'LONDON', 'type': 'LOC'}]"
2,West Indian all-rounder Phil Simmons took four...,"[{'entity': 'West Indian', 'type': 'MISC'}, {'..."


In [4]:
PROMPTS = {
    # --- zero-shot: plain instruction, no examples ---
    "zero_shot_plain": (
        "Named Entity Recognition task.\n"
        "Extract all named entities from the text.\n"
        "Classify each as: PER (person), ORG (organization), LOC (location), MISC (other named entity).\n"
        "The text may be in ALL CAPS. Still extract entities.\n"
        "Respond with a JSON array only. No explanation. No code.\n"
        'Format: [{"entity": "name", "type": "TYPE"}, ...]\n'
        "If no named entities exist, respond: []\n"
        "\n"
        "Text: {text}\n"
        "JSON:"
    ),
    # --- zero-shot: rubric with explicit type definitions ---
    "zero_shot_rubric": (
        "You are an expert Named Entity Recognition system.\n"
        "Label every named entity in the input text using these rules:\n"
        "  PER  = person name (e.g. Barack Obama, Sampras)\n"
        "  ORG  = organization, team, or company (e.g. Reuters, FIFA, Leicestershire)\n"
        "  LOC  = location or place (e.g. London, France, Africa)\n"
        "  MISC = other named entity (e.g. nationality adjective, event name)\n"
        "The text may be in ALL CAPS. Still identify entities.\n"
        'Output ONLY a JSON array: [{"entity": "...", "type": "..."}]. No code. No prose.\n'
        "\n"
        "Text: {text}\n"
        "JSON:"
    ),
    # --- few-shot: 3 caps-style examples that match CoNLL text format ---
    "few_shot_caps": (
        "Extract named entities. Types: PER, ORG, LOC, MISC. Return JSON array only.\n"
        "\n"
        "Text: LEICESTERSHIRE BEAT NOTTINGHAMSHIRE IN CRICKET FINAL .\n"
        'Output: [{"entity": "LEICESTERSHIRE", "type": "ORG"}, {"entity": "NOTTINGHAMSHIRE", "type": "ORG"}]\n'
        "\n"
        "Text: LONDON 1996-08-30\n"
        'Output: [{"entity": "LONDON", "type": "LOC"}]\n'
        "\n"
        "Text: UNITED STATES BEAT GERMANY 2-1 .\n"
        'Output: [{"entity": "UNITED STATES", "type": "LOC"}, {"entity": "GERMANY", "type": "LOC"}]\n'
        "\n"
        "Text: {text}\n"
        "Output:"
    ),
    # --- few-shot: balanced (includes a no-entity example to reduce false positives) ---
    "few_shot_balanced": (
        "Extract named entities from text. Types: PER=person, ORG=organization, LOC=location, MISC=other.\n"
        "Return JSON array only.\n"
        "\n"
        "Text: THE MATCH WAS PLAYED IN PARIS .\n"
        'Output: [{"entity": "PARIS", "type": "LOC"}]\n'
        "\n"
        "Text: JOHN SMITH JOINS REUTERS IN NEW YORK .\n"
        'Output: [{"entity": "JOHN SMITH", "type": "PER"}, {"entity": "REUTERS", "type": "ORG"}, {"entity": "NEW YORK", "type": "LOC"}]\n'
        "\n"
        "Text: THE WEATHER WAS FINE YESTERDAY .\n"
        "Output: []\n"
        "\n"
        "Text: {text}\n"
        "Output:"
    ),
}


def build_prompt(name, text):
    return PROMPTS[name].replace("{text}", text.strip())

In [ ]:
def build_generator(model_name):
    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    load_kwargs = {}
    if device.type == "cuda":
        load_kwargs["torch_dtype"] = torch.float16

    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **load_kwargs)
    model.to(device)
    model.eval()
    return {"tokenizer": tokenizer, "model": model, "device": device}


generators = {}
for model_name in MODEL_NAMES:
    print(f"Loading {model_name} ...")
    generators[model_name] = build_generator(model_name)

print("Loaded models:", list(generators.keys()))

Loading google/flan-t5-small ...


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 1708.57it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading google/flan-t5-base ...


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1017.39it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loaded models: ['google/flan-t5-small', 'google/flan-t5-base']


In [ ]:
ALLOWED_TYPES = {"PER", "ORG", "LOC", "MISC"}


def parse_ner_output(raw_text):
    text = (raw_text or "").strip()
    left = text.find("[")
    right = text.rfind("]")
    candidate = text[left : right + 1] if left != -1 and right != -1 and right > left else text

    parse_ok = True
    entities = []

    try:
        loaded = json.loads(candidate)
        if isinstance(loaded, dict):
            loaded = [loaded]
        if not isinstance(loaded, list):
            loaded = []

        for item in loaded:
            if not isinstance(item, dict):
                continue
            ent = str(item.get("entity", "")).strip()
            typ = str(item.get("type", "")).strip().upper()
            if ent and typ in ALLOWED_TYPES:
                entities.append({"entity": ent, "type": typ})
    except Exception:
        parse_ok = False

    return entities, parse_ok


def to_entity_set(items):
    out = set()
    for item in items:
        ent = " ".join(str(item.get("entity", "")).lower().split())
        typ = str(item.get("type", "")).upper().strip()
        if ent and typ in ALLOWED_TYPES:
            out.add((ent, typ))
    return out


def generate_text(gen, prompt, max_new_tokens=MAX_NEW_TOKENS):
    tokenizer = gen["tokenizer"]
    model = gen["model"]
    model_device = gen["device"]

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    )
    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=NUM_BEAMS,
            early_stopping=True,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


def evaluate_prompt_model(df, prompt_name, model_name):
    gen = generators[model_name]

    tp = fp = fn = 0
    valid = exact = 0
    rows = []

    for row in tqdm(df.itertuples(index=False), total=len(df), desc=f"{model_name} | {prompt_name}"):
        prompt = build_prompt(prompt_name, row.text)
        raw = generate_text(gen, prompt)

        pred_entities, parse_ok = parse_ner_output(raw)
        valid += int(parse_ok)

        gold_set = to_entity_set(row.gold_entities)
        pred_set = to_entity_set(pred_entities)

        tp += len(gold_set & pred_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)
        exact += int(gold_set == pred_set)

        rows.append(
            {
                "text": row.text,
                "gold_entities": row.gold_entities,
                "pred_entities": pred_entities,
                "raw_output": raw,
            }
        )

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

    metrics = {
        "model": model_name,
        "prompt_template": prompt_name,
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "valid_json_rate": round(valid / len(df), 4),
        "exact_sentence_match": round(exact / len(df), 4),
    }

    return metrics, pd.DataFrame(rows)

In [ ]:
run_start = time.perf_counter()

all_metrics = []
all_predictions = {}

for model_name in MODEL_NAMES:
    for prompt_name in PROMPTS.keys():
        metrics_row, pred_df = evaluate_prompt_model(eval_df, prompt_name, model_name)
        all_metrics.append(metrics_row)
        all_predictions[(model_name, prompt_name)] = pred_df

metrics_eval_seconds = time.perf_counter() - run_start

metrics_df = (
    pd.DataFrame(all_metrics)
    .sort_values(by=["f1", "precision", "recall"], ascending=False)
    .reset_index(drop=True)
)

display(metrics_df)

metrics_path = OUTPUT_DIR / "metrics_full_run.csv"
metrics_df.to_csv(metrics_path, index=False)
print("Saved metrics:", metrics_path)
print("Evaluation time (seconds):", round(metrics_eval_seconds, 2))

google/flan-t5-small | zero_shot_plain: 100%|██████████| 80/80 [00:09<00:00,  8.17it/s]
google/flan-t5-small | zero_shot_rubric: 100%|██████████| 80/80 [00:28<00:00,  2.77it/s]
google/flan-t5-small | few_shot_caps: 100%|██████████| 80/80 [00:41<00:00,  1.94it/s]
google/flan-t5-small | few_shot_balanced: 100%|██████████| 80/80 [00:51<00:00,  1.55it/s]
google/flan-t5-base | zero_shot_plain: 100%|██████████| 80/80 [00:43<00:00,  1.85it/s]
google/flan-t5-base | zero_shot_rubric: 100%|██████████| 80/80 [02:16<00:00,  1.71s/it]
google/flan-t5-base | few_shot_caps: 100%|██████████| 80/80 [03:07<00:00,  2.34s/it]
google/flan-t5-base | few_shot_balanced: 100%|██████████| 80/80 [02:29<00:00,  1.87s/it]


,model,prompt_template,precision,recall,f1,valid_json_rate,exact_sentence_match
0,google/flan-t5-small,zero_shot_plain,0.0,0.0,0.0,0.0000,0.125
1,google/flan-t5-small,zero_shot_rubric,0.0,0.0,0.0,0.6500,0.125
2,google/flan-t5-small,few_shot_caps,0.0,0.0,0.0,0.0375,0.125
3,google/flan-t5-small,few_shot_balanced,0.0,0.0,0.0,0.1000,0.125
4,google/flan-t5-base,zero_shot_plain,0.0,0.0,0.0,0.1250,0.125
5,google/flan-t5-base,zero_shot_rubric,0.0,0.0,0.0,0.0000,0.125
6,google/flan-t5-base,few_shot_caps,0.0,0.0,0.0,0.0000,0.125
7,google/flan-t5-base,few_shot_balanced,0.0,0.0,0.0,0.0500,0.125


In [ ]:
shot_df = metrics_df.copy()
shot_df["shot_type"] = shot_df["prompt_template"].apply(
    lambda x: "few-shot" if x.startswith("few_shot") else "zero-shot"
)
shot_df = (
    shot_df.groupby(["model", "shot_type"], as_index=False)[
        ["precision", "recall", "f1", "valid_json_rate", "exact_sentence_match"]
    ]
    .mean()
    .sort_values(by=["model", "shot_type"])
)

print("0-shot vs few-shot summary:")
display(shot_df)

shot_path = OUTPUT_DIR / "shot_summary_full_run.csv"
shot_df.to_csv(shot_path, index=False)
print("Saved shot summary:", shot_path)

# Save logs for report evidence
logs_start = time.perf_counter()
log_examples = eval_df.head(min(5, len(eval_df))).copy()
log_rows = []

for idx, row in log_examples.iterrows():
    for model_name in MODEL_NAMES:
        for prompt_name in PROMPTS.keys():
            prompt = build_prompt(prompt_name, row["text"])
            raw = generate_text(generators[model_name], prompt)
            pred, ok = parse_ner_output(raw)
            log_rows.append(
                {
                    "example_id": idx,
                    "model": model_name,
                    "prompt_template": prompt_name,
                    "parse_ok": ok,
                    "text": row["text"],
                    "gold_entities": row["gold_entities"],
                    "pred_entities": pred,
                    "raw_output": raw,
                }
            )

logs_seconds = time.perf_counter() - logs_start

logs_df = pd.DataFrame(log_rows)
display(logs_df.head(20))

logs_path = OUTPUT_DIR / "ner_prompt_output_logs.csv"
logs_df.to_csv(logs_path, index=False)
print("Saved logs:", logs_path)

calls_metrics = len(eval_df) * len(MODEL_NAMES) * len(PROMPTS)
calls_logs = len(log_examples) * len(MODEL_NAMES) * len(PROMPTS)

avg_seconds_per_metric_call = metrics_eval_seconds / calls_metrics if calls_metrics else 0.0
estimated_full_metrics_seconds = avg_seconds_per_metric_call * full_sentence_count * len(MODEL_NAMES) * len(PROMPTS)
estimated_full_total_seconds = estimated_full_metrics_seconds + logs_seconds

best_row = metrics_df.iloc[0].to_dict()

summary = {
    "use_fast_mode": USE_FAST_MODE,
    "evaluated_sentences": int(len(eval_df)),
    "full_sentences": int(full_sentence_count),
    "models": MODEL_NAMES,
    "prompts": list(PROMPTS.keys()),
    "num_beams": int(NUM_BEAMS),
    "max_new_tokens": int(MAX_NEW_TOKENS),
    "calls_metrics": int(calls_metrics),
    "calls_logs": int(calls_logs),
    "metrics_seconds": round(metrics_eval_seconds, 3),
    "logs_seconds": round(logs_seconds, 3),
    "total_seconds": round(metrics_eval_seconds + logs_seconds, 3),
    "avg_seconds_per_metric_call": round(avg_seconds_per_metric_call, 6),
    "estimated_full_metrics_seconds": round(estimated_full_metrics_seconds, 3),
    "estimated_full_total_seconds": round(estimated_full_total_seconds, 3),
    "estimated_full_total_minutes": round(estimated_full_total_seconds / 60.0, 2),
    "estimated_full_total_hours": round(estimated_full_total_seconds / 3600.0, 2),
    "best_model": best_row["model"],
    "best_prompt": best_row["prompt_template"],
    "best_f1": float(best_row["f1"]),
}

summary_path = OUTPUT_DIR / "run_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved summary:", summary_path)
print(json.dumps(summary, indent=2))

0-shot vs few-shot summary:


,model,shot_type,precision,recall,f1,valid_json_rate,exact_sentence_match
0,google/flan-t5-base,few-shot,0.0,0.0,0.0,0.02500,0.125
1,google/flan-t5-base,zero-shot,0.0,0.0,0.0,0.06250,0.125
2,google/flan-t5-small,few-shot,0.0,0.0,0.0,0.06875,0.125
3,google/flan-t5-small,zero-shot,0.0,0.0,0.0,0.32500,0.125


,example_id,model,prompt_template,parse_ok,text,gold_entities,pred_entities,raw_output
0,0,google/flan-t5-small,zero_shot_plain,False,CRICKET - LEICESTERSHIRE TAKE OVER AT TOP AFTE...,"[{'entity': 'LEICESTERSHIRE', 'type': 'ORG'}]",[],PER
1,0,google/flan-t5-small,zero_shot_rubric,False,CRICKET - LEICESTERSHIRE TAKE OVER AT TOP AFTE...,"[{'entity': 'LEICESTERSHIRE', 'type': 'ORG'}]",[],"[""PER"": ""PER""]"
2,0,google/flan-t5-small,few_shot_caps,False,CRICKET - LEICESTERSHIRE TAKE OVER AT TOP AFTE...,"[{'entity': 'LEICESTERSHIRE', 'type': 'ORG'}]",[],"[""entity"": ""LEICESTERSHIRE"", ""PER"", ""ORG""]"
3,0,google/flan-t5-small,few_shot_balanced,False,CRICKET - LEICESTERSHIRE TAKE OVER AT TOP AFTE...,"[{'entity': 'LEICESTERSHIRE', 'type': 'ORG'}]",[],[CRICKET - LEICESTERSHIRE TAKEOVER AT TOP AFTE...
4,0,google/flan-t5-base,zero_shot_plain,False,CRICKET - LEICESTERSHIRE TAKE OVER AT TOP AFTE...,"[{'entity': 'LEICESTERSHIRE', 'type': 'ORG'}]",[],PER
5,0,google/flan-t5-base,zero_shot_rubric,False,CRICKET - LEICESTERSHIRE TAKE OVER AT TOP AFTE...,"[{'entity': 'LEICESTERSHIRE', 'type': 'ORG'}]",[],"[""entity"": ""..."", ""type"": ""...""]"
6,0,google/flan-t5-base,few_shot_caps,False,CRICKET - LEICESTERSHIRE TAKE OVER AT TOP AFTE...,"[{'entity': 'LEICESTERSHIRE', 'type': 'ORG'}]",[],"[""entity"": ""LEICESTERSHIRE TAKE OVER AT TOP AF..."
7,0,google/flan-t5-base,few_shot_balanced,False,CRICKET - LEICESTERSHIRE TAKE OVER AT TOP AFTE...,"[{'entity': 'LEICESTERSHIRE', 'type': 'ORG'}]",[],"[""entity"": ""LEICESTERSHIRE TAKE OVER AT TOP AF..."
8,1,google/flan-t5-small,zero_shot_plain,False,LONDON 1996-08-30,"[{'entity': 'LONDON', 'type': 'LOC'}]",[],PER
9,1,google/flan-t5-small,zero_shot_rubric,True,LONDON 1996-08-30,"[{'entity': 'LONDON', 'type': 'LOC'}]",[],[]


Saved logs to ner_prompt_output_logs.csv


## Bias and Mitigation

Potential bias sources in prompt-based NER:
- Entity overfitting to frequent names/regions in news.
- Prompt-example imbalance by entity type.
- Domain shift from news text to other domains.

Mitigation:
1. Use balanced few-shot examples.
2. Include no-entity examples (`[]`) to reduce false positives.
3. Evaluate per entity type and per domain.
4. Enforce strict output schema and parse checks.

### Deliverable Coverage
- Multiple prompt designs: yes
- 0-shot vs few-shot: yes
- Multi-model comparison: yes
- Output logs: yes
- Bias discussion: yes